# Notebook 02: HowNet 義原相似度計算

**目標**：使用 OpenHowNet 計算同一批 20 個詞對的語義相似度，
驗證 HowNet 能正確區分 Q1（高相似）與 Q2（低相似）詞對。

> HowNet 相似度基於義原（Sememe）樹的路徑距離，與 LLM cosine similarity 的統計分佈假說完全不同。

In [14]:
import pandas as pd
import numpy as np
import OpenHowNet
from opencc import OpenCC
from pathlib import Path
cc = OpenCC('t2s') # 繁體轉簡體轉換器

# 初始化 HowNet
# 首次執行會下載 HowNet 資料（約 200MB）
OpenHowNet.download()
hownet_dict = OpenHowNet.HowNetDict()
hownet_dict.initialize_similarity_calculation()
print("HowNet 載入完成！")

resources/resources.zip: 72948KB [00:13, 5367.76KB/s]                                                                            


Initializing OpenHowNet succeeded!
Initializing similarity calculation succeeded!
HowNet 載入完成！


In [16]:
# === 使用 OpenHowNet 計算語義相似度 ===
def compute_hownet_similarity(word1: str, word2: str, hownet_dict) -> float:
    """
    呼叫 OpenHowNet 的 calculate_word_similarity() 方法。
    自動將繁體字轉為簡體，若查無此詞回傳 0.0 而非 -1.0。
    """
    try:
        w1_s = cc.convert(word1)
        w2_s = cc.convert(word2)
        score = hownet_dict.calculate_word_similarity(w1_s, w2_s)
        if score is None or score == -1:
            return 0.0
        return float(score)
    except Exception as e:
        print(f"警告: {word1}-{word2} 計算失敗: {e}")
        return 0.0

# 載入刺激詞對組
df = pd.read_csv("../design/stimulus_set.csv")

print("計算 HowNet 語義相似度（繁簡轉換版）...")
df["hownet_similarity"] = df.apply(
    lambda row: compute_hownet_similarity(row["word1"], row["word2"], hownet_dict),
    axis=1
)
print("完成！")

# 同時載入 LLM 結果做對比
llm_results = pd.read_csv("../data/results/01_llm_cosine_scores.csv")[["pair_id","llm_cosine"]]
df = df.merge(llm_results, on="pair_id")
df[["pair_id","word1","word2","quadrant","hownet_similarity","llm_cosine"]]

計算 HowNet 語義相似度（繁簡轉換版）...
完成！


,pair_id,word1,word2,quadrant,hownet_similarity,llm_cosine
0,P01,醫生,護士,Q1,0.792000,0.719874
1,P02,貓,狗,Q1,0.865000,0.683817
2,P03,蘋果,橘子,Q1,1.000000,0.439955
3,P04,鋼琴,吉他,Q1,1.000000,0.633672
4,P05,教師,教授,Q1,0.787375,0.687674
5,P06,醫生,手術刀,Q2,0.406793,0.499631
6,P07,貓,老鼠,Q2,0.574444,0.615028
7,P08,鋼琴,音樂廳,Q2,0.343793,0.442907
8,P09,蘋果,果農,Q2,0.486000,0.546691
9,P10,書,圖書館,Q2,0.318235,0.648320


In [17]:
# === HowNet 象限分組統計 ===
# 【關鍵預測】：HowNet 應能清楚區分 Q1（高分）與 Q2（低分）

from scipy import stats

print("=== HowNet Similarity vs LLM Cosine 各象限均值對比 ===")
comparison = df.groupby("quadrant")[["hownet_similarity","llm_cosine"]].mean().round(4)
print(comparison)

# HowNet: Q1 vs Q2 t-test
q1_hownet = df[df["quadrant"]=="Q1"]["hownet_similarity"].values
q2_hownet = df[df["quadrant"]=="Q2"]["hownet_similarity"].values
t_stat, p_val = stats.ttest_ind(q1_hownet, q2_hownet)
print(f"HowNet Q1 均值: {q1_hownet.mean():.4f}")
print(f"HowNet Q2 均值: {q2_hownet.mean():.4f}")
print(f"t 統計量: {t_stat:.4f}, p 值: {p_val:.4f}")
print(f"結論: {'HowNet 顯著區分 Q1 vs Q2 ✓ 正確！' if p_val < 0.05 else 'HowNet 未能區分，需檢視詞對設計'}")
print(f"[HowNet 成功 vs LLM 失敗]：若 HowNet p<0.05 且 LLM p>0.05，假說 H1 獲得強力支持！")

=== HowNet Similarity vs LLM Cosine 各象限均值對比 ===
          hownet_similarity  llm_cosine
quadrant                               
Q1                   0.8889      0.6330
Q2                   0.4259      0.5505
Q3                   0.8458      0.6560
Q4                   0.4627      0.3869
HowNet Q1 均值: 0.8889
HowNet Q2 均值: 0.4259
t 統計量: 6.9281, p 值: 0.0001
結論: HowNet 顯著區分 Q1 vs Q2 ✓ 正確！
[HowNet 成功 vs LLM 失敗]：若 HowNet p<0.05 且 LLM p>0.05，假說 H1 獲得強力支持！


In [18]:
# 儲存合併結果
output_path = Path("../data/results/02_combined_scores.csv")
df.to_csv(output_path, index=False)
print(f"合併結果已儲存至 {output_path}")

合併結果已儲存至 ../data/results/02_combined_scores.csv
